# Task 3: Symmetric vs Asymmetric INT8 Quantization
Implement and compare symmetric and asymmetric INT8 quantization.

In [1]:
import numpy as np
import pandas as pd

weights = np.array([
    [-1.8, -0.9, 0.0, 0.7, 1.5],
    [-2.4, -0.3, 0.2, 1.1, 2.0]
], dtype=np.float32)

activations = np.array([
    [0.0, 0.3, 0.8, 1.4, 2.1],
    [0.1, 0.6, 1.0, 1.8, 3.2]
], dtype=np.float32)


## Quantization Functions

In [2]:
def symmetric_quantize(tensor):
    x_min, x_max = tensor.min(), tensor.max()
    scale = max(abs(x_min), abs(x_max))/127.0
    if scale==0:
        scale=1.0
    zp=0
    q=np.round(tensor/scale)
    q=np.clip(q,-127,127).astype(np.int8)
    return q,scale,zp

def asymmetric_quantize(tensor):
    x_min,x_max=tensor.min(),tensor.max()
    if np.isclose(x_min,x_max):
        scale=1.0
        zp=0
    else:
        scale=(x_max-x_min)/255.0
        if scale==0:
            scale=1e-12
        zp=round(-128-(x_min/scale))
        zp=int(np.clip(zp,-128,127))
    q=np.round(tensor/scale)+zp
    q=np.clip(q,-128,127).astype(np.int8)
    return q,scale,zp

def dequantize(q,scale,zp):
    return (q.astype(np.float32)-zp)*scale

def metrics(orig,dq,q):
    err=np.abs(orig-dq)
    return {
        "MAE":float(np.mean(err)),
        "MSE":float(np.mean((orig-dq)**2)),
        "Max Err":float(np.max(err)),
        "Sat(min)":int(np.sum(q==-128)),
        "Sat(max)":int(np.sum((q==127)|(q==-127))),
        "Sat(total)":int(np.sum((q==-128)|(q==127)|(q==-127)))
    }


## Apply Both Methods

In [3]:
rows=[]
for name,tensor in [("Weights",weights),("Activations",activations)]:
    for method in ["Symmetric","Asymmetric"]:
        if method=="Symmetric":
            q,s,z=symmetric_quantize(tensor)
        else:
            q,s,z=asymmetric_quantize(tensor)
        dq=dequantize(q,s,z)
        m=metrics(tensor,dq,q)
        rows.append([name,method,s,z,m["MAE"],m["MSE"],m["Max Err"],m["Sat(min)"],m["Sat(max)"],m["Sat(total)"]])

df=pd.DataFrame(rows,columns=["Tensor","Method","Scale","Zero Pt","MAE","MSE","Max Err","Sat(min)","Sat(max)","Sat(total)"])
print(df)


        Tensor      Method     Scale  Zero Pt       MAE       MSE   Max Err  \
0      Weights   Symmetric  0.018898        0  0.003701  0.000022  0.007874   
1      Weights  Asymmetric  0.017255       11  0.003804  0.000021  0.007451   
2  Activations   Symmetric  0.025197        0  0.005276  0.000045  0.011024   
3  Activations  Asymmetric  0.012549     -128  0.002627  0.000011  0.005490   

   Sat(min)  Sat(max)  Sat(total)  
0         0         1           1  
1         1         1           2  
2         0         1           1  
3         1         1           2  


## Side-by-Side Reconstruction

In [4]:
for name,tensor in [("Weights",weights),("Activations",activations)]:
    sq,ss,sz=symmetric_quantize(tensor)
    sdq=dequantize(sq,ss,sz)
    aq,ascl,az=asymmetric_quantize(tensor)
    adq=dequantize(aq,ascl,az)
    print("\n========================")
    print(name)
    print("========================")
    print("Original:\n",tensor)
    print("Symmetric Quantized:\n",sq)
    print("Symmetric Dequantized:\n",sdq)
    print("Asymmetric Quantized:\n",aq)
    print("Asymmetric Dequantized:\n",adq)



Weights
Original:
 [[-1.8 -0.9  0.   0.7  1.5]
 [-2.4 -0.3  0.2  1.1  2. ]]
Symmetric Quantized:
 [[ -95  -48    0   37   79]
 [-127  -16   11   58  106]]
Symmetric Dequantized:
 [[-1.7952756  -0.9070866   0.          0.6992126   1.4929134 ]
 [-2.4        -0.3023622   0.20787401  1.096063    2.0031495 ]]
Asymmetric Quantized:
 [[ -93  -41   11   52   98]
 [-128   -6   23   75  127]]
Asymmetric Dequantized:
 [[-1.7945098  -0.8972549   0.          0.707451    1.5011765 ]
 [-2.3984313  -0.29333332  0.20705882  1.1043137   2.0015686 ]]

Activations
Original:
 [[0.  0.3 0.8 1.4 2.1]
 [0.1 0.6 1.  1.8 3.2]]
Symmetric Quantized:
 [[  0  12  32  56  83]
 [  4  24  40  71 127]]
Symmetric Dequantized:
 [[0.        0.3023622 0.8062992 1.4110236 2.0913386]
 [0.1007874 0.6047244 1.007874  1.7889764 3.2      ]]
Asymmetric Quantized:
 [[-128 -104  -64  -16   39]
 [-120  -80  -48   15  127]]
Asymmetric Dequantized:
 [[0.         0.30117646 0.80313724 1.4054902  2.0956862 ]
 [0.10039216 0.6023529  1.0

## Outlier Experiment

In [5]:
outlier_tensor=np.array([-0.5,-0.2,0.0,0.3,0.7,12.0],dtype=np.float32)
without_outlier=outlier_tensor[:-1]

rows=[]
for version,tensor in [("With outlier",outlier_tensor),("Without outlier",without_outlier)]:
    for method in ["Symmetric","Asymmetric"]:
        if method=="Symmetric":
            q,s,z=symmetric_quantize(tensor)
        else:
            q,s,z=asymmetric_quantize(tensor)
        dq=dequantize(q,s,z)
        err=np.abs(tensor-dq)
        rows.append([version,method,s,z,np.mean(err),np.mean((tensor-dq)**2),np.max(err)])
        print("\n",version,method)
        print("Quantized:",q)
        print("Dequantized:",dq)
        print("Per-element error:",err)

out_df=pd.DataFrame(rows,columns=["Version","Method","Scale","Zero Pt","MAE","MSE","Max Error"])
print("\nComparison Table")
print(out_df)



 With outlier Symmetric
Quantized: [ -5  -2   0   3   7 127]
Dequantized: [-0.47244096 -0.18897638  0.          0.28346455  0.6614173  12.        ]
Per-element error: [0.02755904 0.01102363 0.         0.01653546 0.03858268 0.        ]

 With outlier Asymmetric
Quantized: [-128 -122 -118 -112 -104  127]
Dequantized: [-0.49019608 -0.19607843  0.          0.29411766  0.6862745  12.009804  ]
Per-element error: [0.00980392 0.00392157 0.         0.00588235 0.01372546 0.00980377]

 Without outlier Symmetric
Quantized: [-91 -36   0  54 127]
Dequantized: [-0.5015748 -0.1984252  0.         0.2976378  0.7      ]
Per-element error: [0.00157481 0.0015748  0.         0.00236222 0.        ]

 Without outlier Asymmetric
Quantized: [-128  -64  -22   42  127]
Dequantized: [-0.49882355 -0.19764706  0.          0.3011765   0.7011765 ]
Per-element error: [0.00117645 0.00235294 0.         0.00117648 0.00117654]

Comparison Table
           Version      Method     Scale  Zero Pt       MAE       MSE  \
0    

# Observations

- Symmetric quantization fixes the zero point at 0 and is ideal for weight tensors centered around zero.
- Asymmetric quantization uses a learned zero point and better utilizes the INT8 range for non-negative activations.
- Outliers increase the scale, reducing precision for the remaining values.
- Removing the outlier decreases quantization error and improves reconstruction accuracy.
